# From a pseudobulk demo to a real Task 1 submission

This is an original, synthetic-only walkthrough. It does not contain challenge data, hidden targets, or an organizer notebook. The aim is to make the boundary between a local demonstration and a valid submission format explicit.

Current official sources (check again before upload): [Task 1](https://virtualembryo.ai/challenge/tasks/temporal), [data/panels](https://virtualembryo.ai/challenge/data), [panel index](https://virtualembryo.ai/challenge/panels/index.json), and [submissions](https://virtualembryo.ai/challenge/account/submissions).

## Supported local environment

From the repository root, run `cd modules/task1_walkthrough`, install with `python -m pip install -e '.[notebook]'`, and start `python -m jupyter lab notebooks/task1_baseline_to_submission.ipynb`. This release supports local Jupyter only; Google Colab is not assessed or supported.


As checked on 2026-09-11, Task 1 trains on released E8.5 and E9.5 data; E10.5 is validation and E12.5 is test. There is no E9.25 in the Task-1 single-cell release. E10.5 truth is currently withheld; the official schedule says it will be released at the final phase on 2026-10-20. E12.5 truth is not distributed. The machine-readable `T1:val` index reported 1,000–5,118 cells, while evaluation prose said at least 1,000 with no cap. This companion enforces the dated index as a conservative local preflight policy; it does not resolve the official disagreement, and the upload service/organizers remain authoritative. A real prediction must use the current board's exact ordered `var_names`, required normalization, finite/non-negative two-dimensional `.X`, and no Task-1 spatial coordinates. Structural checks cannot prove the normalization scale.

In [ ]:
# Run from modules/task1_walkthrough after: python -m pip install -e '.[notebook]'
from vec_t1_walkthrough.workflow import (
    build_pseudobulk_shift_prediction,
    make_synthetic_training_inputs,
    validate_task1_prediction,
)


## Offline: preserve the pseudobulk-shift teaching idea

For each `celltype` shared by E8.5 and E9.5, the reference calculation is: `E9.5 cell + (mean(E9.5 type) - mean(E8.5 type))`, then clip to non-negative values. Every E9.5 cell receives its own type's delta, so its within-type variation is retained unless clipping applies. A type only present at E9.5 is copied unchanged; inventing a shift would be unsupported. The method is credited to the published organizer baseline; this companion implementation is not a new predictive method. When a board-bounded subset is needed, this companion samples rows uniformly without replacement with fixed seed 0 and records that output-preparation policy in `<output>.selection.json` rather than prediction metadata. The sidecar has policy, seed, counts, and a digest of selected integer row indices—not cell identifiers. The deterministic synthetic output below is a tiny selected subset for structural inspection, not a competitive baseline claim. It intentionally fails the live board and must not be uploaded.

In [ ]:
early, late, synthetic_panel, synthetic_contract = make_synthetic_training_inputs()
prediction = build_pseudobulk_shift_prediction(
    early, late, synthetic_panel, n_cells=3, contract=synthetic_contract
)
prediction

# The synthetic fixture includes A/B (shared types) and C (E9.5-only).
# The output intentionally contains no celltype field.

In [ ]:
report = validate_task1_prediction(prediction, synthetic_panel, synthetic_contract)
assert report.ok, report.errors
print('Synthetic structural preflight: PASS')
print('Ordered genes:', prediction.var_names.tolist())
print('dtype:', prediction.X.dtype, '| sparse:', type(prediction.X).__name__)

## Online: fetch, record, then validate the live contract

Run the next cell only when you intentionally want a network fetch. It caches `index.json`, the board gene file, and `provenance.json` (URLs, UTC timestamp, and hashes). It never downloads challenge `.h5ad` data. `T1:val` is requested explicitly because this walkthrough is scoped to the E10.5 validation board.

In [ ]:
# Uncomment only for an intentional live official-contract fetch:
# from vec_t1_walkthrough.workflow import fetch_official_contract
# metadata, official_panel = fetch_official_contract('official_contract_cache', board='T1:val')
# print(metadata['key'], len(official_panel), metadata['min_cells'], metadata['max_cells'])

## Local released files: complete the build path

After the intentional fetch, the guarded cell below shows the complete local path: open permitted E8.5/E9.5 files, build with the fetched board order and the dated index-derived 1,000–5,118 local bounds, write a new file and its sampling sidecar, then reopen and preflight it. Evaluation prose currently says at least 1,000 with no cap; this companion policy does not resolve that official disagreement, and the upload service/organizers remain authoritative. Leave the flag `False` for the synthetic notebook run. The command refuses to overwrite an existing output or sidecar. `n_obs` is a board-bounded sample-size choice, not a prediction of embryo size.


In [ ]:
RUN_LOCAL_RELEASED_FILES = False
if RUN_LOCAL_RELEASED_FILES:
    import subprocess
    subprocess.run([
        'vec-t1-walkthrough', 'build',
        '--early', '/path/to/released/E8.5_RNA.h5ad',
        '--late', '/path/to/released/E9.5_RNA.h5ad',
        '--cache', 'official_contract_cache',
        '--n-cells', '1000',
        '--seed', '0',
        '--output', 'artifacts/task1_candidate.h5ad',
    ], check=True)
    subprocess.run([
        'vec-t1-walkthrough', 'validate', 'artifacts/task1_candidate.h5ad',
        '--cache', 'official_contract_cache',
    ], check=True)

## Before the official server

For a permitted real prediction, the guarded path above reads your local files, builds against a freshly cached panel, reopens the output, and validates it. The check intentionally fails on reordered, missing, extra, or duplicate genes rather than silently fixing them. It is a local structural preflight only: it is not the published standalone local scorer (`veckit`) or official challenge server evaluation, and PASS does not mean organizer approval, correct normalization, biological validity, or a useful score. Finite, non-negative raw counts can pass these checks and still be unsuitable for scoring. Task 1 does not require or use spatial coordinates; this companion deliberately rejects `spatial_3D` to keep its minimal output, while current upload prose says `obsm` is ignored. That stricter companion policy is not asserted to mirror server acceptance. As of 2026-09-11, E10.5 validation truth is withheld; E12.5 truth is not distributed. Never manufacture or access held-out truth.
